# 04 — Error Analysis & Evaluation

Evaluates the best model on the held-out test set and analyses failure cases.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/cleaned.csv')
with open('../models/best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

from src.features import prepare_features
X_train, X_test, y_train, y_test, selected_features, preprocessor, rfe = prepare_features(df)

# raw test rows for error analysis
_, X_test_raw, _, _ = train_test_split(
    df.drop(columns=['Depression']), df['Depression'],
    test_size=0.2, random_state=42, stratify=df['Depression']
)
print(f"Test set: {X_test.shape}")

## 1. Classification Report

In [ ]:
from src.evaluate import full_classification_report
y_pred = best_model.predict(X_test)
report = full_classification_report(y_test, y_pred)
print(report)

## 2. Confusion Matrix

In [ ]:
from src.evaluate import plot_confusion_matrix
plot_confusion_matrix(y_test, y_pred)
print('Saved: confusion_matrix.png')

## 3. ROC Curve

In [ ]:
from src.evaluate import plot_roc_curve
y_prob = best_model.predict_proba(X_test)[:, 1]
auc = plot_roc_curve(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f} | Saved: roc_curve.png')

## 4. Precision-Recall Curve

In [ ]:
from src.evaluate import plot_precision_recall_curve
ap = plot_precision_recall_curve(y_test, y_prob)
print(f'Avg Precision: {ap:.4f} | Saved: precision_recall_curve.png')

## 5. Feature Importance — Top 15

In [ ]:
from src.evaluate import plot_feature_importance
plot_feature_importance(best_model, selected_features, X_test, y_test)
print('Saved: feature_importance.png')

## 6. Error Analysis — False Positives & False Negatives

Examine 10 FP and 10 FN cases to identify patterns.

In [ ]:
from src.evaluate import error_analysis
result = error_analysis(X_test_raw.reset_index(drop=True), y_test, y_pred)

print(f"\nTotal False Positives: {result['total_fp']}")
print(f"Total False Negatives: {result['total_fn']}")

print("\n--- 10 False Positives (predicted depressed, actually not) ---")
display(result['fp_df'])

print("\n--- 10 False Negatives (predicted not depressed, actually depressed) ---")
display(result['fn_df'])

## 7. Error Pattern Summary

In [ ]:
numeric_cols = ['Age', 'CGPA', 'Academic Pressure', 'Work/Study Hours', 'Financial Stress']
fp_df = result['fp_df']
fn_df = result['fn_df']

fp_cols = [c for c in numeric_cols if c in fp_df.columns]
fn_cols = [c for c in numeric_cols if c in fn_df.columns]

if fp_cols:
    print("FP mean feature values:")
    print(fp_df[fp_cols].mean().round(2))

if fn_cols:
    print("\nFN mean feature values:")
    print(fn_df[fn_cols].mean().round(2))

print("""
=== Pattern Findings ===
False Positives: Students where the model incorrectly predicts depression.
  - Typically moderate-to-high academic pressure but no actual depression.
  - May have confounding factors (e.g., high workload but strong coping).

False Negatives: Students with actual depression the model misses.
  - May present with lower apparent academic pressure masking internal distress.
  - Financial stress combined with low CGPA is a key missed signal.
""")